In [ ]:
!pip install --upgrade pip
!pip install --upgrade datasets[audio] transformers accelerate evaluate jiwer tensorboard
!pip install --upgrade torch torchvision torchaudio

In [ ]:
from datasets import load_dataset, DatasetDict, Audio
from transformers import (
    WhisperFeatureExtractor, WhisperTokenizer,
    WhisperProcessor, WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments, Seq2SeqTrainer
)
import torch, evaluate, numpy as np, pandas as pd
from dataclasses import dataclass
from typing import Any, Dict, List, Union

MODEL_ID = "openai/whisper-small"
LANGUAGE = "azerbaijani"
TASK = "transcribe"
SEED = 42

N_TRAIN = 459
N_TEST = 57
N_EPOCHS = 3

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
raw = load_dataset("tahmaz/azerbaijani-asr-zenfira_1k", "default", split="train")

dataset_full = raw.rename_column("audio_file", "audio")
dataset_full = dataset_full.rename_column("transcript", "sentence")
dataset_full = dataset_full.cast_column("audio", Audio(sampling_rate=16000))
dataset_full = dataset_full.remove_columns(["duration"])

dataset_full = dataset_full.shuffle(seed=SEED)
small = dataset_full.select(range(N_TRAIN + N_TEST))

split = small.train_test_split(test_size=N_TEST, shuffle=False)
common_voice = DatasetDict({"train": split["train"], "test": split["test"]})

print(len(common_voice["train"]), len(common_voice["test"]))
print(common_voice["train"][0]["sentence"])

In [ ]:
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_ID)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_ID, language=LANGUAGE, task=TASK)
processor = WhisperProcessor.from_pretrained(MODEL_ID, language=LANGUAGE, task=TASK)

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

common_voice = common_voice.map(
    prepare_dataset,
    remove_columns=common_voice.column_names["train"],
    num_proc=1
)
print(common_voice)

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print("ok")

In [ ]:
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {"wer": 100 * wer_metric.compute(predictions=pred_str, references=label_str)}

In [ ]:
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model = model.to(device)
print(MODEL_ID)

In [ ]:
from transformers import EarlyStoppingCallback

In [ ]:
model.config.dropout = 0.1
model.config.attention_dropout = 0.1
model.config.apply_spec_augment = True
model.config.mask_time_prob = 0.05
model.config.mask_feature_prob = 0.05

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir                  = "./whisper-small-az",
    per_device_train_batch_size = 8,
    per_device_eval_batch_size  = 8,
    gradient_accumulation_steps = 2,
    weight_decay                = 0.01,
    learning_rate               = 3e-5,
    warmup_steps                = 50,
    num_train_epochs            = N_EPOCHS,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    greater_is_better           = False,
    metric_for_best_model       = "wer",
    predict_with_generate       = True,
    generation_max_length       = 225,
    logging_steps               = 5,
    save_total_limit            = 2,
    report_to                   = ["tensorboard"],
    logging_dir                 = "./logs",
    fp16                        = torch.cuda.is_available(),
    dataloader_num_workers      = 0,
    push_to_hub                 = False,
)

trainer = Seq2SeqTrainer(
    args             = training_args,
    model            = model,
    train_dataset    = common_voice["train"],
    eval_dataset     = common_voice["test"],
    data_collator    = data_collator,
    compute_metrics  = compute_metrics,
    processing_class = processor.feature_extractor,
    callbacks        = [EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
train_result = trainer.train()

trainer.save_model()
trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)
trainer.save_state()

print(trainer.state.best_model_checkpoint)

In [ ]:
log_history = trainer.state.log_history
train_steps = [(e["step"], e["loss"]) for e in log_history if "loss" in e and "eval_loss" not in e]
eval_epochs = [(e["epoch"], e["eval_loss"], e["eval_wer"]) for e in log_history if "eval_loss" in e and "eval_wer" in e]

df_eval = pd.DataFrame(eval_epochs, columns=["Epoch", "Val Loss", "Val WER (%)"])
print(df_eval.to_string(index=False))

best_idx = df_eval["Val WER (%)"].idxmin()
print(df_eval["Val WER (%)"].min(), df_eval.loc[best_idx, "Epoch"])

In [ ]:
import string
from jiwer import wer as jwer, cer as jcer

base_model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID).to(device).eval()
ft_model = WhisperForConditionalGeneration.from_pretrained("./whisper-small-az").to(device).eval()

def normalize(text):
    text = text.lower().strip()
    text = text.translate(str.maketrans('', '', string.punctuation + '«»—–'))
    return " ".join(text.split())

def transcribe(mdl, audio_dict):
    feats = processor.feature_extractor(
        audio_dict["array"], sampling_rate=audio_dict["sampling_rate"], return_tensors="pt"
    ).input_features.to(device)
    with torch.no_grad():
        fids = processor.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)
        ids = mdl.generate(feats, forced_decoder_ids=fids)
    return processor.tokenizer.batch_decode(ids, skip_special_tokens=True)[0].strip()

test_raw = dataset_full.select(range(N_TRAIN, N_TRAIN + N_TEST))

results = []
for i in range(len(test_raw)):
    row = test_raw[i]
    ref = row["sentence"].strip()
    if not ref:
        continue
    try:
        hyp_b = transcribe(base_model, row["audio"])
        hyp_f = transcribe(ft_model, row["audio"])
    except Exception as e:
        print(i, e)
        continue

    rn = normalize(ref)
    bn = normalize(hyp_b) or "<bos>"
    fn = normalize(hyp_f) or "<bos>"

    results.append(dict(
        ref=ref, hyp_b=hyp_b, hyp_f=hyp_f,
        bwer=jwer(rn, bn), fwer=jwer(rn, fn),
        bcer=jcer(rn, bn), fcer=jcer(rn, fn)
    ))

    if (i + 1) % 10 == 0:
        bw = np.mean([r["bwer"] for r in results]) * 100
        fw = np.mean([r["fwer"] for r in results]) * 100
        print(i + 1, round(bw, 2), round(fw, 2))

df = pd.DataFrame(results)

In [ ]:
mbw = df["bwer"].mean() * 100
mfw = df["fwer"].mean() * 100
mbc = df["bcer"].mean() * 100
mfc = df["fcer"].mean() * 100

summary = pd.DataFrame([
    {"Model": "Baza (whisper-small)",    "WER": round(mbw, 2), "CER": round(mbc, 2), "dWER": "-",              "dCER": "-"},
    {"Model": "Fine-tuned (whisper-az)", "WER": round(mfw, 2), "CER": round(mfc, 2), "dWER": round(mbw-mfw, 2), "dCER": round(mbc-mfc, 2)},
])

print(summary.to_string(index=False))

arrow = "azaldi" if mbw > mfw else "artdi"
print(round(abs(mbw - mfw), 2), arrow)

Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

steps_x = [s[0] for s in train_steps]
steps_y = [s[1] for s in train_steps]
ep_x = df_eval["Epoch"].tolist()
val_loss = df_eval["Val Loss"].tolist()
val_wer = df_eval["Val WER (%)"].tolist()

best_ep = df_eval.loc[df_eval["Val WER (%)"].idxmin(), "Epoch"]
best_wer = df_eval["Val WER (%)"].min()

fig = plt.figure(figsize=(16, 13))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.50, wspace=0.35)

C = {
    "train": "#E07B54", "val": "#4A90D9", "best": "#27AE60",
    "base":  "#E74C3C", "ft":  "#2ECC71", "cer":  "#9B59B6"
}

ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(steps_x, steps_y, color=C["train"], lw=1.8, alpha=0.9, label="Train Loss")
ax1.fill_between(steps_x, steps_y, alpha=0.12, color=C["train"])
ax1.set_title("Train Loss (step uzre)", fontsize=12, fontweight="bold")
ax1.set_xlabel("Step")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(True, ls="--", alpha=0.4)
ax1.spines[["top", "right"]].set_visible(False)

steps_per_epoch = max(1, len(steps_x) // N_EPOCHS)
train_loss_ep = [
    np.mean(steps_y[i * steps_per_epoch:(i + 1) * steps_per_epoch])
    for i in range(len(ep_x))
]

ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(ep_x, train_loss_ep, color=C["train"], lw=2, marker="o", ms=5, label="Train Loss")
ax2.plot(ep_x, val_loss,      color=C["val"],   lw=2, marker="s", ms=5, label="Val Loss")
ax2.axvline(best_ep, color=C["best"], ls="--", lw=1.5, label="best ep " + str(int(best_ep)))
ax2.fill_between(ep_x,
    [min(tl, vl) for tl, vl in zip(train_loss_ep, val_loss)],
    [max(tl, vl) for tl, vl in zip(train_loss_ep, val_loss)],
    alpha=0.08, color="red", label="gap")
ax2.set_title("Train vs Val Loss", fontsize=12, fontweight="bold")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.set_xticks(ep_x)
ax2.legend(fontsize=8)
ax2.grid(True, ls="--", alpha=0.4)
ax2.spines[["top", "right"]].set_visible(False)

ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(ep_x, val_wer, color=C["val"], lw=2, marker="o", ms=7, markerfacecolor="white", markeredgewidth=2)
ax3.axvline(best_ep, color=C["best"], ls="--", lw=1.5, label=str(round(best_wer, 1)) + "% ep" + str(int(best_ep)))
ax3.scatter([best_ep], [best_wer], color=C["best"], zorder=5, s=80)
for x, y in zip(ep_x, val_wer):
    ax3.annotate(str(round(y, 1)), (x, y), xytext=(0, 8), textcoords="offset points", ha="center", fontsize=8)
ax3.set_title("Validation WER per Epoch", fontsize=12, fontweight="bold")
ax3.set_xlabel("Epoch")
ax3.set_ylabel("WER (%)")
ax3.set_xticks(ep_x)
ax3.legend(fontsize=9)
ax3.grid(True, ls="--", alpha=0.4)
ax3.spines[["top", "right"]].set_visible(False)

ax4 = fig.add_subplot(gs[1, 1])
cats  = ["Baza", "Fine-tuned"]
wers  = [mbw, mfw]
cers  = [mbc, mfc]
x_pos = np.arange(2)
bw_   = 0.3
b1 = ax4.bar(x_pos - bw_/2, wers, bw_, color=C["base"], label="WER (%)", edgecolor="white", lw=0.8)
b2 = ax4.bar(x_pos + bw_/2, cers, bw_, color=C["cer"],  label="CER (%)", edgecolor="white", lw=0.8)
for b in list(b1) + list(b2):
    ax4.text(b.get_x() + b.get_width()/2, b.get_height() + 0.4,
             str(round(b.get_height(), 1)) + "%", ha="center", va="bottom", fontsize=9)
ax4.set_title("Baza vs Fine-tuned WER & CER", fontsize=12, fontweight="bold")
ax4.set_xticks(x_pos)
ax4.set_xticklabels(cats, fontsize=10)
ax4.set_ylabel("Xeta (%)")
ax4.legend()
ax4.grid(True, axis="y", ls="--", alpha=0.4)
ax4.spines[["top", "right"]].set_visible(False)

plt.suptitle(MODEL_ID + "  train: " + str(N_TRAIN) + "  test: " + str(N_TEST), fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("training_results.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
df["improvement"] = df["bwer"] - df["fwer"]

print("En cox komek etdiyi 3:")
for i, row in enumerate(df.sort_values("improvement", ascending=False).head(3).itertuples(), 1):
    print(i, round(row.improvement * 100, 1))
    print(row.ref)
    print(row.hyp_b)
    print(row.hyp_f)

print("\nEn az komek etdiyi 3:")
for i, row in enumerate(df.sort_values("improvement").head(3).itertuples(), 1):
    print(i, round(row.improvement * 100, 1))
    print(row.ref)
    print(row.hyp_b)
    print(row.hyp_f)